# Projet : Les accidents corporels routiers en France 

## Introduction

Nous voulons étudier les accidents corporels de la route en France. Pour ce faire nous avons récoltés les données issus de la base de donnée BAAC (Bases de données annuelles des accidents corporels de la circulation routière). Cette base comporte quatre fichier principaux, avec un fichier par année. Nous voulons avoir plusieurs années, donc nous décidons de récolter les fichiers des années 2022, 2023 et 2024, cela nous fait donc 12 fichiers pour notre import de donnée. 
Les quatres fichiers sont :
- Les caractéristiques de l'accidents
- Le lieux de l'accidents
- Les véhicules impliqués dans l'accidents
- Les usagers impliqués dans l'accidents

Dans chacun de ces fichiers, nous avons plusieurs variables qui caractérisent l'accidents et les informations auxiliaires. Nous avons notamment la variables de gravités des blessures des usagers impliqués dans les accidents, une des variables d'interet principale que nous etudierons.

Nous avons répondre à la problématique suivante : Quels facteurs influencent la gravité des accidents corporels de la route en France, peut-on prédire cette gravité en tenant compte des caractéristiques de l'accident (environnementales, géographique...) ?
Tout d'abord, nous vous presenterons les données à travers le traitement de celles-ci. Puis nous répondrons au sujet avec des analyses descriptives et de la modélisation. Pour finir, nous conclurons. 

In [48]:
import matplotlib.pyplot as plt
import numpy as np
from great_tables import GT
import random
from sklearn.metrics import confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.model_selection import GridSearchCV

# Traitement
from src.donnees import import_donnees, renomer_cle_jointure, concatenation_annees
from src.nettoyage import recodage, mapping_renommer_colonnes, colonnes_a_supprimer, création_age_usager, jointure, creation_mois_num, rajout_colonnes

# Analyse descriptive
from src.fct_carto import creation_df_carte, carte_departement
from src.fonction_analyse import effectif_frequence, tableau_propre_effectif_frequence, chi2_cramer, tableau_propre_cramer, evolution_mensuelle, nb_accidents_par, tab_cont_grav, bar_chart

# Modelisation
from src.fct_modelisation import y_x_train_test, importance_variable, importance_variable_gravite
from src.fct_sortie_propre import beau_report, matrice_confusion, beau_importances, beau_importance_gravite, belle_colonnes, belle_head

In [47]:
import importlib
import src.fonction_analyse

importlib.reload(src.fonction_analyse)

<module 'src.fonction_analyse' from '/home/onyxia/work/Projet_pythonDS/src/fonction_analyse.py'>

In [35]:
import importlib
import src.fct_sortie_propre

importlib.reload(src.fct_sortie_propre)



<module 'src.fct_sortie_propre' from '/home/onyxia/work/Projet_pythonDS/src/fct_sortie_propre.py'>

## I - Traitement des données

Nous avons réalisé des analyses préliminaires sur nos données (disponible dans le fichier pre-processing.ipynb), cela nous a permis de relever certains problèmes dans les données que nous avons pris en compte pour le nettoyage de nos données ci-dessous (comme les na, les noms de colonnes, les doublons ...)

In [36]:
donnees_completes = import_donnees()

donnees_completes["caract"][22] = renomer_cle_jointure(donnees_completes["caract"][22], "Num_Acc", "Accident_Id")

/home/onyxia/work/Projet_pythonDS/src/donnees.py:8: DtypeWarning: Columns (0: lartpc) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(nom_fichier_csv, sep=';', encoding='UTF-8')
/home/onyxia/work/Projet_pythonDS/src/donnees.py:8: DtypeWarning: Columns (0: nbv) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(nom_fichier_csv, sep=';', encoding='UTF-8')


In [37]:
# CARACTERISTIQUE
df_caract = concatenation_annees(donnees_completes, "caract")
df_caract = creation_mois_num(df_caract)
# LIEUX
df_lieux = concatenation_annees(donnees_completes, "lieux")
# VEHICULE
df_vehicule = concatenation_annees(donnees_completes, "vehicule")
# USAGER
df_usager = concatenation_annees(donnees_completes, "usager")

In [38]:
mappings = mapping_renommer_colonnes()

In [39]:
# recodage des noms des colonnes 
df_caract_recoder = recodage(df_caract, mappings["caract"])
df_lieux_recoder = recodage(df_lieux, mappings["lieux"])
df_vehicule_recoder = recodage(df_vehicule, mappings["vehicule"])
df_usager_recoder = recodage(df_usager, mappings["usager"])

# supprimer les doublons de corrections des données dans le fichier lieux 
df_lieux_recoder = df_lieux_recoder.drop_duplicates(subset="Num_Acc", keep="last")

# age pour les usagers
df_usager_recoder = création_age_usager(df_usager_recoder)

In [40]:
df_final = jointure(df_caract_recoder, df_lieux_recoder, df_vehicule_recoder, df_usager_recoder)

In [41]:
colonnes_a_supprimer = colonnes_a_supprimer()
df_final = df_final.drop(columns=colonnes_a_supprimer, errors="ignore")
df_final = df_final.dropna(subset=["id_usager"])

In [42]:
df_final = rajout_colonnes(df_final)

In [43]:
#df_final.columns
belle_colonnes(df_final)

Num_Acc,jour,mois,an,hrmn,lum,dep,agg,int,atm,col,lat,long,mois_num,catr,circ,vosp,prof,plan,surf
infra,situ,vma,id_vehicule,catv,obs,obsm,choc,manv,id_usager,catu,grav,sexe,trajet,secu1,age,date,jour_semaine,hr,nan


In [44]:
#df_final.head()
belle_head(df_final)

,Num_Acc,jour,mois,an,hrmn,lum,dep,agg,int,atm,col,lat,long,mois_num,catr,circ,vosp,prof,plan,surf,infra,situ,vma,id_vehicule,catv,obs,obsm,choc,manv,id_usager,catu,grav,sexe,trajet,secu1,age,date,jour_semaine,hr
0,202400000001,25,mars,2024,07:40,Crépuscule ou aube,70,Hors agglomération,Hors intersection,Brouillard - fumée,Deux véhicules - frontale,"47,56277000","6,75832000",3,Route départementale,Bidirectionnelle,Sans objet,Plat,En courbe à gauche,Normale,Aucun,Sur chaussée,90,155 781 758,VL,Sans objet,Véhicule,Avant,Déporté,203 988 581,Conducteur,Blessé hospitalisé,Homme,Domicile - École,Ceinture,23,2024-03-25 00:00:00,Lundi,07
1,202400000001,25,mars,2024,07:40,Crépuscule ou aube,70,Hors agglomération,Hors intersection,Brouillard - fumée,Deux véhicules - frontale,"47,56277000","6,75832000",3,Route départementale,Bidirectionnelle,Sans objet,Plat,En courbe à gauche,Normale,Aucun,Sur chaussée,90,155 781 759,PL,Sans objet,Véhicule,Avant droit,Manœuvre d’évitement,203 988 582,Conducteur,Indemne,Homme,Utilisation professionnelle,Ceinture,29,2024-03-25 00:00:00,Lundi,07
2,202400000002,20,mars,2024,15:05,Plein jour,21,En agglomération,Intersection en T,Temps éblouissant,Autre collision,"47,02109000","4,83755000",3,Voie communale,À sens unique,Sans objet,Plat,Partie rectiligne,Autre,Aucun,Sur chaussée,30,155 781 757,VU,Sans objet,Piéton,Avant gauche,Tournant,203 988 579,Piéton,Blessé hospitalisé,Femme,Promenade - loisirs,Aucun équipement,99,2024-03-20 00:00:00,Mercredi,15
3,202400000002,20,mars,2024,15:05,Plein jour,21,En agglomération,Intersection en T,Temps éblouissant,Autre collision,"47,02109000","4,83755000",3,Voie communale,À sens unique,Sans objet,Plat,Partie rectiligne,Autre,Aucun,Sur chaussée,30,155 781 757,VU,Sans objet,Piéton,Avant gauche,Tournant,203 988 580,Conducteur,Indemne,Homme,Utilisation professionnelle,Ceinture,39,2024-03-20 00:00:00,Mercredi,15
4,202400000003,22,mars,2024,19:30,Crépuscule ou aube,15,Hors agglomération,Hors intersection,Normale,Autre collision,"44,90238400","2,49641800",3,Voie communale,Bidirectionnelle,Sans objet,Plat,Partie rectiligne,Normale,Aucun,Sur accotement,50,155 781 756,Voiturette,Mobilier urbain,Piéton,Avant,Sans changement de direction,203 988 574,Passager,Blessé léger,Femme,Promenade - loisirs,Non déterminable,19,2024-03-22 00:00:00,Vendredi,19


Certaines variables contiennent des valeurs non renseigné, pour certaines nous décidons donc de supprimer les lignes comportant des valeurs pour d'autre nous enlevons la colonne.
Pour faire notre choix nous considérons plusieurs éléments :
- enlever les lignes : si les NA sont représneter en faible quantités dans la colonnes, si la colonne est importantes. Par exemple pour la colonne gravité de l'accidnet nous avons 359 NA sur 377 638 cela représente une faible proportion des usagers de plus la colonne gravité nous est totalement indispensable pour faire nos analyses car nous voulons regarder les gravités des différents accidents 
- enlever les colonnes : si il y a beaucoup de NA et que la colonne n'est pas forcement pertinentes pour nos analyses 

Les variables avaient une repartition assez fidèle sur la gravité des accidents. Sauf quelques variables qui n'avait que des indemnes ou tout sauf des tué, mais cela ne pose pas de pb car bcp dans les données. En revanche, pour la variable trajet (29% de NA), il y a trop de gravité Tué pour pouvoir supprimé les na de cette variable, nous la gardons pour l'analyser en revanche la modelisation ne la prendra pas en compte. De même poiur circ.

In [45]:
# suppression des NA 
df_final = df_final.dropna(subset=["lum", "int", "atm", "col", "prof", "plan", "surf", "situ", "catv", "obs", "obsm", "choc", "sexe", "age", "secu1"])

## II - Analyse Descriptive des accidents corporels en France

### 1. Vue d'ensemble

In [53]:
effectif_frequence(df_final, "grav")

,grav,effectif,frequence
0,Indemne,NaN,NaN
1,Blessé léger,NaN,NaN
2,Blessé hospitalisé,NaN,NaN
3,Tué,NaN,NaN
4,Total,0.0,0.0


In [ ]:
tableau_propre_effectif_frequence(eff_freq)

GT(_tbl_data=                 grav  effectif  frequence
0             Indemne       NaN        NaN
1        Blessé léger       NaN        NaN
2  Blessé hospitalisé       NaN        NaN
3                 Tué       NaN        NaN
4               Total       0.0        0.0, _body=<great_tables._gt_data.Body object at 0x7f17fee33590>, _boxhead=Boxhead([ColInfo(var='grav', type=<ColInfoTypeEnum.default: 1>, column_label='Gravité', column_align='left', column_width=None), ColInfo(var='effectif', type=<ColInfoTypeEnum.default: 1>, column_label='Effectif', column_align='right', column_width=None), ColInfo(var='frequence', type=<ColInfoTypeEnum.default: 1>, column_label='Fréquence', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7f1807b05eb0>, _spanners=Spanners([]), _heading=Heading(title='Gravité des accidents pour les usagers', subtitle='Répartition des effectifs et des fréquences pour la variable « gravité »', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7f17fede9220>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7f17fede94f0>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7f17ff341fd0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x7f17fee03e50>, <great_tables._gt_data.FormatInfo object at 0x7f17fede97c0>], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_bottom_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_left_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_left_width=OptionsInfo(scss

In [50]:
chi2_cramer = chi2_cramer(df_final, "grav")[["variable", "v_cramer"]]
tableau_propre_cramer(chi2_cramer)

GT(_tbl_data=                                           variable  v_cramer
0                            Équipement de sécurité    0.2671
1                             Catégorie du véhicule    0.2443
2                             Catégorie de l'usager    0.1669
3                          En ou hors agglomération    0.1663
4             Manoeuvre principale avant l'accident    0.1605
5                         Département de l'accident    0.1548
6                              Obstacle fixe heurté    0.1547
7                                 Type de collision    0.1511
8                            Obstacle mobile heurté    0.1443
9                           Situation de l'accident    0.1284
10                            Point de choc initial    0.1274
11                                   Type de trajet    0.1244
12                               Catégorie de route    0.1033
13                                 Sexe de l'usager    0.0914
14                            Régime de circulation    0.0862
15                                    Tracé en plan    0.0712
16             Luminosité et conditions d'éclairage    0.0684
17                                     Intersection    0.0564
18                                            Heure    0.0546
19  Déclivité de la route à l'endroit de l'accident    0.0416
20                    Existence d'une voie réservée    0.0394
21                        Conditions atmosphériques    0.0329
22                               Jour de la semaine    0.0315
23                    Infrastructure ou aménagement    0.0270
24                               État de la surface    0.0263
25                               Mois de l'accident    0.0218, _body=<great_tables._gt_data.Body object at 0x7f18041f4f50>, _boxhead=Boxhead([ColInfo(var='variable', type=<ColInfoTypeEnum.default: 1>, column_label='Variable', column_align='left', column_width=None), ColInfo(var='v_cramer', type=<ColInfoTypeEnum.default: 1>, column_label='V de Cramèr', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7f17fedf62c0>, _spanners=Spanners([]), _heading=Heading(title='Association entre la gravité et différentes variables', subtitle='\n                Résultats des V de Cramèr\n                entre la gravité et différentes variables qualitatives\n            ', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7f17ff9d1d00>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7f17fee15d00>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7f17fedf5f30>, _formats=[<great_tables._gt_data.FormatInfo object at 0x7f17fee15480>], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', t

### 2. Évolution des accidents dans le temps

#### Vue d'ensemble

Il s'agit ici d'étudier les accidents sur la période 2022-2024 pour déterminer s'il y a eu une éventuelle augmentation ou diminution, et si les accidents présentent une cyclicité.

In [ ]:
nb_accidents_par(df_final, "an", "Année")

On remarque déjà que le nombre d'accidents est comparable sur les trois années. On peut ensuite s'intéresser à l'évolution mois par mois.

In [ ]:
evolution_mensuelle(df_final)

On constate d'une part que si à l'année le nombre d'accidents ne semble pas présenter de tendance, il varie d'un mois à l'autre en restant compris entre 3600 et 5300. D'autre part, le nombre d'accidents est généralement reflété par le nombre d'usagers impliqués (deux à trois usagers pour un accident en moyenne), lui-même reflété par le nombre de victimes (les usagers blessés ou tués). Les usagers indemnes représentent 41,23 % de ceux impliqués dans les accidents corporels de la base de données.

Évolutions hebdomadaire et horaire

In [ ]:
ordre = ["Lundi", "Mardi", "Mercredi", "Jeudi", "Vendredi", "Samedi", "Dimanche"]
nb_accidents_par(df_final, "jour_semaine", "Jour de la semaine", ordre, True)

In [ ]:
nb_accidents_par(df_final, "hr", "Heure")

Le vendredi est le jour de la semaine qui cumule le plus d'accidents (26804), et le dimanche le moins (20102). Le reste de la semaine comporte des valeurs aux alentours de 22000 ou 23000.

On observe deux modes aux heures de pointes, à 8 et 17 heures. Les accidents sont ensuite plus nombreux de jour que de nuit, et laissent supposer que leur nombre reflète au moins en partie l'intensité du trafic.

Les accidents sont d'ailleurs plus nocturnes le week-end que la semaine, et la distribution moins répartie autour des heures de pointes :

In [ ]:
nb_accidents_par(df_final[df_final["jour_semaine"].isin(["Samedi", "Dimanche"])], "hr", "Heure")

### 3. Nombre et gravité des accidents selon les facteurs extérieurs

In [ ]:
ordre_colonnes = [
    "Indemne", 
    "Blessé léger", 
    "Blessé hospitalisé", 
    "Tué"
]

#### Conditions atmosphériques et météorologiques

La majeure partie des accidents a lieu dans des conditions de conduite « normales » : sans défaut de visibilité dû à la météo ni problème d'éclairage, et sur des routes non altérées par l'environnement. La rareté des phénomènes inhabituels se retranscrit par une faible occurrence dans les données. Mises à part les conditions d'éclairage, les conditions qui dépendent directement ou indirectement de l'environnement sont de l'ordre des centaines seulement lorsqu'elles sortent de l'ordinaire.

In [ ]:
ordre_lignes_lum  = [
    "Plein jour",
    "Crépuscule ou aube",
    "Nuit sans éclairage public",
    "Nuit avec éclairage public non allumé",
    "Nuit avec éclairage public allumé"
]

nb_accidents_par(df_final, "lum", "Condition d'éclairage", ordre_lignes_lum, True)

In [ ]:
ordre_lignes_atm = [
    "Normale", 
    "Pluie légère", 
    "Pluie forte", 
    "Neige - grêle", 
    "Brouillard - fumée", 
    "Vent fort - tempête", 
    "Temps éblouissant", 
    "Temps couvert", 
    "Autre"
]
nb_accidents_par(df_final, "atm", "Condition atmosphérique", ordre_lignes_atm, True)

Il est en outre possible de s'intéresser à la gravité des accidents pour ces différentes modalités :

In [ ]:
tc_lum_grav = tab_cont_grav(df_final, "lum", ordre_lignes_lum, ordre_colonnes)
bar_chart(tc_lum_grav, "Conditions d'éclairage", "Répartition de la gravité de l'accident selon la luminosité")

In [ ]:
tc_atm_grav = tab_cont_grav(df_final, "atm", ordre_lignes_atm, ordre_colonnes)
bar_chart(tc_atm_grav, "Conditions atmosphériques", "Répartition de la gravité de l'accident selon les conditions atmosphériques")

Les situations où la visibilité est affectée semblent être les plus fatales. 6,3 % des usagers impliqués dans un accident de nuit sans éclairage public ont trouvé la mort (4,7 % pour les accidents avec éclairage public non allumé). Ce chiffre n'est que de 1,6 % avec ledit éclairage allumé, et de 2,3 % en plein jour.

De surcroît, le brouillard et/ou la fumée constituent la modalité associée à la plus forte mortalité parmi les conditions atmosphériques directes avec 6,4 % de tués.

In [ ]:
ordre_lignes_surf = [
   "Normale",
   "Mouillée",
   "Flaques",
   "Inondée",
   "Enneigée",
   "Boue",
   "Verglacée",
   "Corps gras - Huile",
   "Autre"
]

tc_surf_grav = tab_cont_grav(df_final, "surf", ordre_lignes_surf, ordre_colonnes)
bar_chart(tc_surf_grav, "Surface au sol", "Répartition de la gravité de l'accident selon la surface")

Bien que ces deux surfaces concernent un faible nombre d'accidents dans les données, la proportion de tués est plus grande lorsque les accidents ont lieu sur des surfaces inondées (87 accidents) ou boueuses (143) par rapport aux autres situations : 9,6 % et 7,6 % respectivement.

La part de blessés hospitalisés est particulièrement importante dans ce second cas (36,4 %, soit 10 points de plus que les autres types de surface spécifiés).

#### Conditions routières

On traitera ici des conditions de conduite associées à la route elle-même ou au type de circulation.

In [ ]:
ordre_lignes = [
    "Autoroute",
    "Route nationale",
    "Route départementale",
    "Voie communale",
    "Hors réseau public",
    "Parc de stationnement ouvert à la circulation publique",
    "Route de métropole urbaine",
    "Autre"
]
nb_accidents_par(df_final, "catr", "Catégorie de route", ordre_lignes, True)

In [ ]:
tc_catr_grav = tab_cont_grav(df_final, "catr", ordre_lignes, ordre_colonnes)
bar_chart(tc_catr_grav, "Catégorie de route", "Répartition de la gravité de l'accident selon la catégorie de route")

Les accidents ont surtout lieu sur le réseau public (voie communale et route départementale en tête), même si les routes hors réseau ont une proportion de tués légèrement plus importante.

In [ ]:
ordre_lignes_circ = [
    "À sens unique",
    "Bidirectionnelle",
    "À chaussées séparées",
    "Avec voies d'affectation variable",
    "NA"
]

nb_accidents_par(df_final.fillna({"circ": "NA"}), "circ", "Régime de circulation", ordre_lignes_circ, True)


Les voies bidirectionnelles sont les plus représentées parmi les observations. Malheureusement, les valeurs manquantes sont assez nombreuses pour cette variable, mais leur répartition de la gravité associée ne diffère pas drastiquement de celle des autres modalités :

In [ ]:

tc_circ_grav = tab_cont_grav(df_final.fillna({"circ": "NA"}), "circ", ordre_lignes_circ, ordre_colonnes)
bar_chart(tc_circ_grav, "Régime de circulation", "Répartition de la gravité de l'accident selon le régime de circulation")

#### Caractéristiques de l'accident

Nous nous intéressons ici à quelques facteurs venant caractériser l'accident : type de collision, endroit du choc, objet percuté...

In [ ]:
ordre_lignes_col = [
    "Deux véhicules - frontale",
    "Deux véhicules - par l'arrière",
    "Deux véhicules - par le côté",
    "Trois véhicules - en chaîne",
    "Trois véhicules - collisions multiples",
    "Autre collision",
    "Sans collision"
]

nb_accidents_par(df_final, "col", "Type de collision", ordre_lignes_col, True)

tc_col_grav = tab_cont_grav(df_final, "col", ordre_lignes_col, ordre_colonnes)
bar_chart(tc_col_grav, "Type de collision", "Répartition de la gravité de l'accident selon le type de collision")

L'absence de collision est le cas le plus fatal (6,4 %) parmi les situations où le type de collision est spécifié, et c'est aussi le cas avec le moins d'indemnes : 16,8 % seulement. Elle est suivie par la catégorie « Autre », puis par la collision frontale entre deux véhicules, et le cas de collisions multiples entre trois véhicules. On remarque que pour les collisions entre trois véhicules, plus de la moitié des usagers impliqués dans l'accident s'en sortent indemnes.

In [ ]:
ordre_lignes_choc = [
"Aucun",
"Avant",
"Avant droit",
"Avant gauche",
"Arrière",
"Arrière droit",
"Arrière gauche",
"Côté droit",
"Côté gauche",
"Chocs multiples (tonneaux)"
]

tc_choc_grav = tab_cont_grav(df_final, "choc", ordre_lignes_choc, ordre_colonnes)
bar_chart(tc_choc_grav, "Choc initial", "Répartition de la gravité de l'accident selon l'endroit du choc initial")

Les chocs à l'avant concernent la grande majorité des accidents (62543 accidents dans la base de données). Les chocs multiples (tonneaux) sont les moins représentés (3147) mais aussi ceux qui comportent le plus de blessés et tués (83,5 % des usagers impliqués) et sont mortels pour 6,8 % des usagers. Les cas avec les conséquences les plus graves sont ensuite ceux où il n'y a techniquement pas de choc (5 % de tués), les chocs complètement à l'avant, les chocs latéraux, les avants gauche et droit, et enfin les chocs arrières.

In [ ]:
ordre_lignes_catv = [
    "Indéterminable",
    "Bicyclette",
    "Cyclomoteur",
    "Voiturette",
    "Scooter",
    "Motocyclette",
    "VL",
    "VU",
    "PL",
    "Tracteur routier",
    "Tramway",
    "Engin spécial",
    "Tracteur agricole",
    "Quad",
    "Autobus",
    "Autocar",
    "Train",
    "3RM",
    "EDP à moteur",
    "EDP sans moteur",
    "VAE",
    "Autre véhicule"
]

tc_catv_grav = tab_cont_grav(df_final, "catv", ordre_lignes_catv, ordre_colonnes)
bar_chart(tc_catv_grav, "Catégorie de véhicule", "Répartition de la gravité de l'accident selon la catégorie du véhicule")

Enfin, la proportion de blessés et tués est également affectée par la catégorie du ou des véhicules impliqués dans l'accident. Plus ce dernier est robuste, plus la proportion d'indemnes est grande. Ce sont globalement les accidents impliquant des deux-roues qui ont le plus de chance de comporter des blessés.

#### Caractéristiques de l'usager

Les positions et équipements des individus impliqués dans un accident de la route peuvent affecter leurs chances de survie et leurs risques de blessure.

In [ ]:
ordre_lignes_catu = [
    "Conducteur",
    "Passager",
    "Piéton"
]

tc_catu_grav = tab_cont_grav(df_final, "catu", ordre_lignes_catu, ordre_colonnes)
bar_chart(tc_catu_grav, "Catégorie d'usager", "Répartition de la gravité de l'accident selon la catégorie de l'usager")


La gravité de l'accident est drastiquement différente entre les usagers véhiculés et les usagers piétons. Ces derniers sont beaucoup plus vulnérables et ne s'en sortent indemnes que dans 3,2 % des cas, contre près de 40 % pour les deux autres catégories.

In [ ]:
ordre_lignes_secu1 = [
    "Aucun équipement",
    "Ceinture",
    "Casque",
    "Dispositif enfants",
    "Gilet réfléchissant",
    "Airbag (2RM/3RM)",
    "Gants (2RM/3RM)",
    "Gants + Airbag (2RM/3RM)",
    "Non déterminable",
    "Autre"
]

nb_accidents_par(df_final, "secu1", "Équipement de sécurité", ordre_lignes_secu1, True)

tc_secu1_grav = tab_cont_grav(df_final, "secu1", ordre_lignes_secu1, ordre_colonnes)
bar_chart(tc_secu1_grav, "Équipement de sécurité", "Répartition de la gravité de l'accident selon la catégorie de l'usager")


L'équipement de sécurité est un autre facteurs qui affecte grandement les proportions de la gravité. Ici, toutes les barres du graphique ne sont pas interprétables : les airbags étaient peu représentés, et les proportions qui ont utilisé cette modalité manquent de généralité. 

Du reste, on peut remarquer que le risque de blessure ou de mort diminue de moitié entre l'absence de sécurité et l'équipement d'au moins une ceinture. Cela est cependant nuancé par le fait que l'absence de sécurité peut dans une partie de ces cas être associée à des véhicules deux roues ne comportant pas de ceintures. 

C'est également sous ce prisme que l'on observe la vulnérabilité associée à l'équipement du casque et des gants : ils concernent avant tout les accidents deux-roues, plus mortels que les accidents des véhicules plus imposants.

### 4. Dans quel département français les accidents sont-ils les plus graves ? Un peu de cartographie

Nous souhaitons voir si certains departements ont plus d'accidents grave que d'autres ou inversement. 
Par accident, il peut y avoir plusieurs victime, nous considérons donc le nombre de victime et non le nombre d'accident :

In [ ]:
# nombre d'accident
print("nombre d'accidents : ", df_final["Num_Acc"].nunique())
# nombre de victime 
print("nombre de victimes : ", df_final["id_usager"].nunique())

In [ ]:
# recupération du nombre de victime par departement et gravité
df_victime_dep = df_final.groupby("dep").size().reset_index(name="nb").sort_values(by="nb")
print(df_victime_dep)

Comme on peut le voir ci-dessus, le nombre d'accident par departement est très inégale, on observe notamment beaucoup d'accidnet dans les départements en ile de France, cela peut s'expiquer du fait qu'il ya plus d'habitants aussi. 
Du a cela, nous allons étudier la proportion des blessures d'accident plutot que le nombre qui est trop disproportionné en fonction des départements.

In [ ]:
print(df_final["grav"].unique())

In [ ]:
df_victimes_tot = creation_df_carte(df_final)

In [ ]:
carte_departement(df_victimes_tot)

In [ ]:
df_victimes_blessure = creation_df_carte(df_final, blessure=True)

In [ ]:
carte_departement(df_victimes_blessure, blessure="Tué")

In [ ]:
carte_departement(df_victimes_blessure, blessure="indemne")

Les victimes tués par un accdient sont une faible proportion des victimes, ou on en voit le plus dans la diagonal du vide 
En revanche, les accidents indemne sont plus vers le nord.

In [ ]:
df_victimes_an = creation_df_carte(df_final, an=True)

In [ ]:
carte_departement(df_victimes_an, an=2023)

In [ ]:
df_victimes_an_blessure = creation_df_carte(df_final, an=True, blessure=True)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 8))

carte_departement(df_victimes_an_blessure, blessure="tué", an=2022, ax=axes[0])
carte_departement(df_victimes_an_blessure, blessure="tué", an=2023, ax=axes[1])
carte_departement(df_victimes_an_blessure, blessure="tué", an=2024, ax=axes[2])

fig.suptitle("Comparaison des tués par département – 2022 / 2023 / 2024", fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 8))

carte_departement(df_victimes_an_blessure, blessure="Blessé hospitalisé", an=2022, ax=axes[0])
carte_departement(df_victimes_an_blessure, blessure="Blessé hospitalisé", an=2023, ax=axes[1])
carte_departement(df_victimes_an_blessure, blessure="Blessé hospitalisé", an=2024, ax=axes[2])

fig.suptitle("Comparaison des Blessé hospitalisé par département – 2022 / 2023 / 2024", fontsize=18)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 8))

carte_departement(df_victimes_an_blessure, blessure="Blessé léger", an=2022, ax=axes[0])
carte_departement(df_victimes_an_blessure, blessure="Blessé léger", an=2023, ax=axes[1])
carte_departement(df_victimes_an_blessure, blessure="Blessé léger", an=2024, ax=axes[2])

fig.suptitle("Comparaison des Blessé léger par département – 2022 / 2023 / 2024", fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 8))

carte_departement(df_victimes_an_blessure, blessure="Indemne", an=2022, ax=axes[0])
carte_departement(df_victimes_an_blessure, blessure="Indemne", an=2023, ax=axes[1])
carte_departement(df_victimes_an_blessure, blessure="Indemne", an=2024, ax=axes[2])

fig.suptitle("Comparaison des Indemne par département – 2022 / 2023 / 2024", fontsize=18)
plt.tight_layout()
plt.show()

# III - Classification de la gravité d'un accident par un Random Forest

Nous nous interressons à la prédiction de la gravité des blessures des usagers d'un accident de la route. Pour ce faire nous allons étudier :
- la variable de gravité (variables catégorielle à expliquer) : 1 = Indemne, 2 = Blessé léger, 3 = Blessé hospitalisé, 4 = Tué
- un Random Forest, nous permettant de classifier nos données
- des facteurs clés pour classifier la gravité d'accidents : choc avant / arrière / latéral, âge de l'usager, sexe, port de la ceinture / casque, luminosité, état de la chaussée, type de route (autoroute, nationale, départementale), zone (urbaine / rurale), département, heure, jour de la semaine ...

Les valeurs manquantes des variables que nous allons utiliser ci-après ont déjà été géré (c'est à dire supprimé) dans la première phase de ce projet. Les variables comportant encore des valeurs NA dans nos données ne seront pas utilisé pour les modèles et supprimé en amont.

In [ ]:
# Graine de reproductibilité
np.random.seed(66)
random.seed(66)

In [ ]:
# Codage de la variable cible en numérique
grav_dict = {
    "Indemne":1,
    "Blessé léger":2,
    "Blessé hospitalisé":3,
    "Tué":4,
}
df_final = recodage(df_final, {"grav": grav_dict})

### Encodage des variables explicatives 

Nous identifions les variables les plus pertinantes pour notre analyses. Pour l'heure nous faisons un regroupement en fonction des heures : Nuit (00–06), Matin (06–12),Après‑midi (12–18), Soir (18–24). 
Puis nous séparons les valeurs en jeu d'entrainement et de test, sans oublier de coder les variables catégorielles. (toute sauf l'âge)

In [ ]:
# Séparation du jeu de donnée en y et X puis en jeu d'entrainement et de test en passant par l'encodage
# suppression NA de manv
df_final = df_final.dropna(subset=["manv"])
X_train, X_test, y_train, y_test = y_x_train_test(df_final, "grav", ["mois", "an", "lum", "dep", "agg", "int", "surf", "catv", "obs","obsm",
                                                                     "choc", "sexe", "situ", "secu1", "age", "atm", "col", "catr", "prof",
                                                                     "plan", "manv", "catu", "jour_semaine"])

### Entrainement du modèle 

Les classes de gravité "Tué" sont moins nombreuses que les autres, mais comme on veut classifier correctement les classes et notamment celle-ci car il est important de pouvoir identifier les accidents avec une gravités élevé, il nous faut donc attribuer des poids aux classes.

In [ ]:
poids = {
    "1": 1,   # indemne
    "2": 1.07,   # blessé léger
    "3": 2.77,   # blessé hospitalisé
    "4": 15.4   # tué
}

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight=poids,
    random_state=66,
    n_jobs=-1
)

rf.fit(X_train, y_train)


### Evaluation et importance des facteurs 

In [ ]:
y_pred = rf.predict(X_test)
beau_report(y_test, y_pred)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
matrice_confusion(cm)

Les classes 1 et 2 sont bien prédites, pour les classes 3 et 4 beaucoup moins de bonnes prédictions, cela est du au fait que les données comporte moins d'accident sur ces gravités de blessures, le 3 et 4 sont tué et blessures hospitalisé. Surtout au niveau des "Tué", le modele ne retrouve de 0.22 bonne valeurs, ce qui est plutot faible. Même si le modèle a une accuracy globale correct de 66%, nous voulons notamment prédire les tués pour voir ce qui est le plus dangereux, donc ici notre modèle n'est pas le plus performant. 
La matrice de confusion confirme les résultats.

Nous testons alors, un second modèle qui permet de mieux prendre en compte les classes rares, comme c'est le cas ici pour les gravités de blessures "Tué" :

In [ ]:
brf = BalancedRandomForestClassifier(
    n_estimators=400,
    sampling_strategy="auto",
    random_state=66,
    n_jobs=-1
)

brf.fit(X_train, y_train)


In [ ]:
y_pred_brf = brf.predict(X_test)
beau_report(y_test, y_pred_brf)

In [ ]:
cm = confusion_matrix(y_test, y_pred_brf)
matrice_confusion(cm)

Ce modèle détecte nettement mieux les classes rares, en revanche, il y a aussi une baisse de la présicion car le modèle se trompe plus souvent quand il classifie en accident "Tué", alors qu'il ne le sont pas.

### GridSearchCV - tentative d'amélioration du modèle avec l'optimisation des paramètres


In [ ]:
param_grid = {
    "n_estimators": [100],   
    "max_depth": [None, 12, 20], 
    "class_weight": [
        {"1":1, "2":1, "3":2, "4":6},
        {"1":1, "2":1, "3":3, "4":15},
        "balanced_subsample"
    ]
}

In [ ]:
rf_grid = RandomForestClassifier(
    random_state=66,
    n_jobs=1
)

grid = GridSearchCV(
    estimator=rf_grid,
    param_grid=param_grid,
    scoring="recall_macro",
    refit="recall_macro",
    cv=3,
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)


In [ ]:
print("Meilleurs paramètres :", grid.best_params_)
print("Meilleurs score :", grid.best_score_)

Selon la gridsearch les meilleurs parametres sont ci-dessus. Nous allons donc refaire un modele avec ces paramètres :

In [ ]:
rf_opti = RandomForestClassifier(
    n_estimators=400,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    random_state=66,
    n_jobs=-1
)

rf_opti.fit(X_train, y_train)

y_pred = rf_opti.predict(X_test)
beau_report(y_test, y_pred)


In [ ]:
cm = confusion_matrix(y_test, y_pred)
matrice_confusion(cm)

En optimisant, les paramètres et en utilisant le scoring "recall_macro", notre modèle détecte bien les accidents "Tué". Mais la précision est en baisse. 
Pour analyser l'importance des variables, nous allons choisir le modèle réalisé avec balancedRandomForest qui est meilleur pour la détéction des classes rares que notre modèle réalisé avec gridsearch. En revanche, ce modèle est moins bon globalement que le premier modèle RandomForest que nous avons effectuer. Cependant comme nous voulons identifier les gravités (sans manquer les classes les plus rares), nous garderons celui-ci.

In [ ]:
beau_importances(importance_variable(brf, X_train))

In [ ]:
# Corrélation entre variables et probabilité de la gravité "Tué"
beau_importance_gravite(importance_variable_gravite(brf, X_test, 3))

In [ ]:
# Corrélation entre variables et probabilité de la gravité "Indemne"
beau_importance_gravite(importance_variable_gravite(brf, X_test, 0))

In [ ]:
# Corrélation entre variables et probabilité de la gravité "Bléssé léger"
beau_importance_gravite(importance_variable_gravite(brf, X_test, 1))

In [ ]:
# Corrélation entre variables et probabilité de la gravité "Bléssé hospitalisé"
beau_importance_gravite(importance_variable_gravite(brf, X_test, 2))

On observe donc plusiuers variables qui sont corrélés au fait d'avoir des blessures plus grave que d'autres :
- le port de la ceinture de sécurité : une corrélation de -0.32 pour les "Tué" et pour aucun_équipement 0.2868 - le fait de ne pas avoir d'équipement de sécurité favorise le fait d'avoir des accidents grave (c'est l'inverse pour les "Indemne")
- le type de route, hors agglomération et route départementale favorise un accident à haut risque
- la catégorie de véhicule : les véhicule lourds (VL) plus de risque de blessures lègéres.
